# Capability 17: Conversation memory optimization for long-running sessions

3/3 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 17`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap17_memory_optimization.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: short conversation (3 turns) -- should NOT have a rolling summary yet

**✅ PASS**

In [ ]:
_ = ask('What was South America revenue in 2024?', '01a')
_ = ask('And in 2025?', '01b')
_ = ask('What drove that change?', '01c')

01a
USER: What was South America revenue in 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: South America revenue in 2024 was **$12,423 million USD** (~$12.4 billion), based on the quarterly KPI data.

Would you like a quarterly breakdown, or a comparison with 2023 revenue for South America?

01b
USER: And in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: In 2025, **South America** generated **$11,954 million (USD)** in revenue, based on the aggregated quarterly data.

01c
USER: What drove that change?

[intent=data_query | sub_agents=['unstructured']]
[citations: ['DOC-015', 'DOC-012', 'DOC-008', 'DOC-003']]

AGENT: " includes only the below. But user asked "What drove that change?" We need answer as if we have context. We can infer change is revenue change. We should say "Based on the evidence available, the specific drivers of the South America revenue change in 2025 aren't explicitly attributed. The data confirms the change itself..."

Need decide whe

## 02: long conversation (8 turns) -- SHOULD trigger a rolling summary

**✅ PASS**

In [ ]:
_ = ask('What was South America revenue in 2024?', '02a')
_ = ask('And in 2025?', '02b')
_ = ask('What drove that change?', '02c')
_ = ask('Any related earnings commentary?', '02d')
_ = ask('What about its EBITDA margin there?', '02e')
_ = ask('How does that compare to Middle Americas?', '02f')
_ = ask('What was the volume trend for South America in 2025?', '02g')
_ = ask("And North America's revenue for the same year?", '02h')

02a
USER: What was South America revenue in 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: South America revenue in 2024 was **$12,423 million USD** (~$12.4 billion), based on the quarterly KPI data.

Would you like a quarterly breakdown, or a comparison with 2023 revenue for South America?

02b
USER: And in 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: In 2025, **South America** generated **$11,954 million (USD)** in revenue, based on the aggregated quarterly data.

02c
USER: What drove that change?

[intent=data_query | sub_agents=['structured', 'unstructured']]
[citations: ['DOC-015', 'DOC-012', 'DOC-008', 'DOC-003']]

AGENT: ## What drove South America's revenue change (FY2024 → FY2025)

**Headline:** South America revenue fell **$469M**, from **$12,423M** in 2024 to **$11,954M** in 2025.

Decomposing that change into volume vs. price/mix:

| Driver | Effect (USD M) |
|---|---|
| Volume effect | **-432.4** |
| Price/mix effect | **-36.6** |
| **Tota

## 03: long conversation stays coherent/doesn't crash once summarized

**✅ PASS**

In [ ]:
_ = ask('What was North America revenue in 2024?', '03a')
_ = ask('And 2025?', '03b')
_ = ask('What was the EBITDA margin both years?', '03c')
_ = ask('Compare that to EMEA.', '03d')
_ = ask("What's driving EMEA's numbers?", '03e')
_ = ask('Any recent commentary on EMEA?', '03f')
_ = ask("What about Asia Pacific's revenue in 2025?", '03g')
_ = ask("Summarize everything we've discussed so far.", '03h')

03a
USER: What was North America revenue in 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America revenue in 2024 was **$14,655 million (USD)**. [structured data]

If useful, I can look at year-over-year growth, a monthly/quarterly breakdown, or revenue by brand/market within North America.

03b
USER: And 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: For **2025**, revenue in **North America** was **$14,208 million USD** (≈ $14.2B), based on the available KPI data.

Would you like a breakdown by quarter or by brand for 2025?

03c
USER: What was the EBITDA margin both years?

[intent=comparison | sub_agents=['structured']]

AGENT: **North America – EBITDA margin (computed from quarterly data)**

| Year | EBITDA Margin |
| --- | --- |
| 2024 | 32.7% |
| 2025 | 33.0% |

That's a modest improvement of roughly **30 basis points** year over year.

One caveat on methodology: these figures are not reported annual margins — they are computed as the sum of 